# Code de préparation des données pour projet 1 INSEE (innatio)

05/12/2025 - Master 2 SPE La Rochelle - UE Data to Information.

Auteur : Gustave Jouis

Génère le fichier de données `data_etrangers.geojson` qui est utilisé par mon_graphique.py

In [ ]:
from flask import Flask, render_template, request, make_response
import pandas as pd
import geopandas as gpd
import json
from shapely.geometry import shape
import folium
from sqlalchemy import create_engine
import logging

## Importation des données :

1. Imprter les données sources avec les noms des EPCI (Aurélie)

In [ ]:
data = pd.read_csv(r"C:\Users\gusta\Documents\Master GEEL\M2 GEEL\Python\Projet\INAT_NAT_EPCI_REGION.csv", sep=';')
#importer le fichier avec le nom des EPCI
epci=pd.read_excel(r"C:\Users\gusta\Documents\Master GEEL\M2 GEEL\Python\Projet\EPCI_au_01-01-2025.xlsx")
epci.head()
data['EPCI'] = data['EPCI'].astype(str)
epci['EPCI'] = epci['EPCI'].astype(str)
#jointure des deux dataframes pour avoir les noms des EPCI
data = epci[['EPCI', 'LIBEPCI']].merge(
    data, on='EPCI', how='left'
)
data=data.rename(columns={'LIBEPCI':'nom'})
data

In [ ]:
print(data['EPCI'].dtype)
print(epci['EPCI'].dtype)

2. Ajouter les données géographiques à partir du fichier json de Dina

In [ ]:
#importer le json et le convertir
geo = pd.read_json(r"C:\Users\gusta\Documents\Master GEEL\M2 GEEL\Python\Projet\referents-contours-epci.json")
geo = geo.rename(columns={'geo_shape': 'geometry'})
geo["geometry"] = geo["geometry"].apply(lambda x: shape(json.loads(x)))
geo["geometry"] = geo["geometry"].apply(shape)

#créer le geodataframe
geo_epci = gpd.GeoDataFrame(geo, geometry="geometry", crs=4326)
geo_epci = geo_epci.rename(columns={'code_epci': 'EPCI'})
#affichage pour vérification
# geo_epci.plot()
# print(geo_epci.head())
geo_epci['EPCI'] = geo_epci['EPCI'].astype(str)
data['EPCI'] = data['EPCI'].astype(str)
#ajouter les coordonnées aux données
data_geo = geo_epci[['EPCI', 'geometry']].merge(data, on='EPCI', how='left')
data_geo.head()

#transformer total_s en numérique
data_geo['total_s'] = pd.to_numeric(data_geo['total_s'], errors='coerce')

#supprimer les doublons
data_geo = data_geo.sort_values(by='EPCI').drop_duplicates(subset=['EPCI', 'nom', 'NAT_rec3'])
data_geo = gpd.GeoDataFrame(
    data_geo, 
    geometry='geometry',  # colonne contenant les objets Shapely
    crs='EPSG:4326'       # définir le CRS si connu
)

In [ ]:
data_geo.plot()

## Création de la carte

1. Exploration de la création d'une carte follium pour une région

In [ ]:
#filtrer les données pour la région normandie
# data_geo_normandie = data_geo[data_geo['region_name'] == 'Normandie']
data_geo_normandie = data_geo
data_geo_normandie

In [ ]:
# convertir la géométrie en WKT (Excel ne supporte pas les objets shapely)
#data_geo["geometry"] = data_geo["geometry"].apply(lambda geom: geom.wkt)

# sauvegarder en Excel
#data_geo.to_excel("data_geo.xlsx", index=False)

In [ ]:
# # Calcul du nombre d'etrangers par EPCI en Normandie
# part_etranger_normandie = data_geo_normandie[data_geo_normandie['NAT_rec3'].isin(['etrangers', 'Tous'])]
# part_etranger_normandie = part_etranger_normandie.pivot_table(
#     index=['EPCI', 'nom', 'geometry'],  # colonnes qui restent fixes
#     columns='NAT_rec3',                # valeur unique par colonne
#     values='total_s'                  # colonne à mettre dans les nouvelles colonnes
# ).reset_index()
# part_etranger_normandie['Pct_Etranger'] = (part_etranger_normandie['etrangers'] / part_etranger_normandie['Tous']) * 100
# part_etranger_normandie

all_epci = data_geo_normandie[['EPCI', 'nom', 'geometry']].drop_duplicates()

pivot = data_geo_normandie[data_geo_normandie['NAT_rec3'].isin(['etrangers', 'Tous'])].pivot_table(
    index=['EPCI', 'nom', 'geometry'],
    columns='NAT_rec3',
    values='total_s',
    aggfunc='sum',
    fill_value=0
).reset_index()

part_etranger_normandie = all_epci.merge(
    pivot, on=['EPCI', 'nom', 'geometry'], how='left'
)

part_etranger_normandie['etrangers'] = part_etranger_normandie['etrangers'].fillna(0)
part_etranger_normandie['Tous'] = part_etranger_normandie['Tous'].fillna(0)

part_etranger_normandie['Pct_Etranger'] = (
    part_etranger_normandie['etrangers'] / part_etranger_normandie['Tous'] * 100
)

In [ ]:
min_values = part_etranger_normandie['Tous'].min()
print("Minimums :\n", min_values)

In [ ]:
#Determiner les nationalités étrangeres les plus présentes par EPCI

#filtrer les données pour les nationalités étrangères
# etrangers_normandie = data_geo_normandie[data_geo_normandie['NAT_rec3']!='Français']


# etrangers_normandie = etrangers_normandie.sort_values(['EPCI', 'total_s'], ascending=[True, False])
# top3_per_epci = etrangers_normandie.groupby('EPCI').head(3).reset_index(drop=True)
# top3_per_epci['NAT_rec3'] = top3_per_epci['NAT_rec3'].fillna('').astype(str)
# top3_str = (
#     top3_per_epci
#     .groupby('EPCI')['NAT_rec3']
#     .apply(lambda x: ', '.join(x))
#     .reset_index()
# )
# top3_str = top3_str.rename(columns={'NAT_rec3': 'top3'})
# top3_str
# top3_str = top3_str.rename(columns={'top3': 'top3_nationalites'})  # nouveau nom

# 1️⃣ Filtrer uniquement les nationalités étrangères réelles
etrangers = data_geo_normandie[
    (data_geo_normandie['NAT_rec3'] != 'Français') &
    (data_geo_normandie['NAT_rec3'] != 'Tous') &
    (data_geo_normandie['NAT_rec3'] != 'immigres') &
    (data_geo_normandie['NAT_rec3'] != 'etrangers') &
    (data_geo_normandie['NAT_rec3'].notna())&
    (data_geo_normandie['INAT_BIS'] == 'Etranger')
]

# 2️⃣ Trier par total_s décroissant **à l’intérieur de chaque EPCI**
etrangers = etrangers.groupby('EPCI', group_keys=False).apply(
    lambda x: x.sort_values('total_s', ascending=False)
)

# 3️⃣ Prendre les 3 nationalités les plus représentées par EPCI
top3 = etrangers.groupby('EPCI').head(3).reset_index(drop=True)

top3_str = top3.merge(
    data_geo_normandie[['EPCI', 'region_name']].drop_duplicates(),
    on='EPCI',
    how='left'
)

# 4️⃣ Concaténer les nationalités en une seule chaîne
top3_str = (
    top3
    .groupby(['EPCI', 'region_name'])
    .apply(lambda g: ', '.join(g['NAT_rec3']))
    .reset_index(name='top3')
)

top3_str
#fusionner avec le dataframe des pourcentages
top3_str = top3_str.rename(columns={'top3': 'top3_nationalites'})

part_etranger_normandie = part_etranger_normandie.merge(
    top3_str,
    on='EPCI',
    how='left'  # garder tous les EPCI même s'il n'y a pas de top3
)
part_etranger_normandie['Pct_Etranger'] = part_etranger_normandie['Pct_Etranger'].round(1)
part_etranger_normandie['Pct_Etranger_str'] = part_etranger_normandie['Pct_Etranger'].astype(str) + '%'

part_etranger_normandie





In [ ]:
top3_str

In [ ]:
part_etranger_normandie


In [ ]:
import geopandas as gpd
import folium
import branca.colormap as cm

from shapely import wkt

gdf_normandie = gpd.GeoDataFrame(
    part_etranger_normandie,
    geometry='geometry',
    crs='EPSG:4326'
)
# 1️⃣ S'assurer que c'est un GeoDataFrame
gdf_normandie = gpd.GeoDataFrame(
    part_etranger_normandie, 
    geometry='geometry', 
    crs='EPSG:4326'
)

gdf_normandie['Pct_Etranger'] = gdf_normandie['Pct_Etranger'].fillna(0) 



# 2️⃣ Optionnel : simplifier la géométrie pour Folium (rapide)
gdf_normandie['geometry'] = gdf_normandie['geometry'].simplify(tolerance=0.001)

# 3️⃣ Créer une palette automatique
colormap = cm.linear.YlOrRd_09.scale(
    gdf_normandie['Pct_Etranger'].min(),
    gdf_normandie['Pct_Etranger'].max()
)
colormap.caption = "Pourcentage d'étrangers (%)"

# 4️⃣ Fonction de style
def style_function(feature):
    pct = feature['properties']['Pct_Etranger']
    return {
        'fillColor': colormap(pct),
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.7,
    }

# 5️⃣ Créer la carte
m = folium.Map(location=[49.0, 0.5], zoom_start=7)

# Ajouter les polygones
folium.GeoJson(
    gdf_normandie,
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(
        fields=['nom', 'Pct_Etranger_str', 'top3_nationalites'],
        aliases=['EPCI:', 'Pct étrangers:', 'Nationalités étrangeres les plus présentes :'],
        localize=True
    )
).add_to(m)

# Ajouter la légende
colormap.add_to(m)

# 6️⃣ Afficher la carte
#m

m.save("carte_pct_etrangers_normandie.html")



In [ ]:
print(type(part_etranger_normandie["geometry"].iloc[0]))


Créer la page web avec menu déroulant :

In [ ]:
data_geo


In [ ]:
#exporter données au format geojson
part_etranger_normandie = gpd.GeoDataFrame(
    part_etranger_normandie,
    geometry="geometry",
    crs="EPSG:4326"   # ou le CRS de tes données
)
part_etranger_normandie.to_file(r"C:\Users\gusta\Documents\Master GEEL\M2 GEEL\Python\Projet\data_etrangers.geojson", driver="GeoJSON")

In [ ]:
data = gpd.read_file(r"C:\Users\gusta\Documents\Master GEEL\M2 GEEL\Python\Projet\data_etrangers.geojson")  # ou ton fichier
print(data.columns)